# AirScope: Air Pollution Analytics in Indian Cities

This notebook explores historical air-quality data, cleans it, and creates graphical analysis for Pune, Mumbai, Delhi, and Bengaluru where data is available.

**Dataset:** Kaggle *Air Quality Data in India* by Rohan Rao. Download `city_day.csv` and place it in the `data/` folder. Source: https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india

**Important:** This dataset contains historical observations (commonly 2015–2020), not current air quality. Results describe this dataset's coverage and should not be treated as live AQI.

In [ ]:
%pip install pandas numpy matplotlib seaborn plotly


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (11, 6)
DATA_PATH = Path('data/city_day.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Place city_day.csv inside the project data/ folder. See README.md for download instructions.')
df = pd.read_csv(DATA_PATH)
print('Rows, columns:', df.shape)
display(df.head())
print('\nColumns:', df.columns.tolist())

## 1. Data quality and preparation

We parse dates, inspect missing values, and standardize city names. Missing pollutant values are not automatically filled because doing so can distort pollution patterns.

In [ ]:
df.columns = [c.strip() for c in df.columns]
if 'Date' not in df.columns or 'City' not in df.columns:
    raise ValueError('Expected columns named City and Date. Check that you loaded city_day.csv.')
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['City'] = df['City'].astype('string').str.strip()
df = df.dropna(subset=['Date', 'City']).copy()
for col in df.columns:
    if col not in ['City', 'Date', 'AQI_Bucket']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.sort_values(['City', 'Date']).reset_index(drop=True)
print('Date range:', df['Date'].min().date(), 'to', df['Date'].max().date())
print('Cities:', df['City'].nunique())
display(df.isna().sum().sort_values(ascending=False).to_frame('missing_count'))


## 2. Choose cities and key columns

We use the four-city comparison if those cities exist in the file. If one is absent, the code reports it and continues with available cities.

In [ ]:
preferred_cities = ['Pune', 'Mumbai', 'Delhi', 'Bengaluru']
available_cities = set(df['City'].dropna().unique())
cities = [c for c in preferred_cities if c in available_cities]
missing_cities = [c for c in preferred_cities if c not in available_cities]
if not cities:
    cities = sorted(df['City'].dropna().unique())[:6]
print('Cities included:', cities)
if missing_cities:
    print('Not found in dataset:', missing_cities)
analysis = df[df['City'].isin(cities)].copy()
pollutants = [c for c in ['PM2.5', 'PM10', 'NO2', 'NO', 'NOx', 'NH3', 'CO', 'SO2', 'O3'] if c in analysis.columns]
print('Available pollutant columns:', pollutants)


## 3. Graph 1 — Number of observations by city

This checks whether city comparisons have similar amounts of data. Unequal coverage can affect comparisons.

In [ ]:
plt.figure()
sns.countplot(data=analysis, x='City', order=analysis['City'].value_counts().index)
plt.title('Available Daily Records by City')
plt.xlabel('City'); plt.ylabel('Number of daily records'); plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

## 4. Graph 2 — Average AQI by city

The bars compare mean recorded AQI values. Interpret alongside the time range and data coverage; an average can hide short pollution episodes.

In [ ]:
if 'AQI' in analysis.columns:
    city_aqi = analysis.groupby('City', observed=True)['AQI'].mean().sort_values(ascending=False).dropna()
    display(city_aqi.to_frame('mean_AQI'))
    plt.figure(); sns.barplot(x=city_aqi.index, y=city_aqi.values)
    plt.title('Mean Recorded AQI by City'); plt.xlabel('City'); plt.ylabel('Mean AQI')
    plt.xticks(rotation=20); plt.tight_layout(); plt.show()
else:
    print('AQI column is unavailable; skip this chart.')

## 5. Graph 3 — AQI distribution by city

Box plots show the median, spread, and potential high-value outliers. Outliers are observations worth investigating, not necessarily errors.

In [ ]:
if 'AQI' in analysis.columns:
    plt.figure(); sns.boxplot(data=analysis, x='City', y='AQI')
    plt.title('Distribution of Daily AQI by City'); plt.xticks(rotation=20)
    plt.tight_layout(); plt.show()

## 6. Graph 4 — AQI over time

Daily values can be noisy. A 30-day rolling mean makes broader patterns easier to see without removing the original observations.

In [ ]:
if 'AQI' in analysis.columns:
    daily = analysis.dropna(subset=['AQI']).sort_values('Date').copy()
    daily['AQI_30d'] = daily.groupby('City')['AQI'].transform(lambda s: s.rolling(30, min_periods=7).mean())
    plt.figure(figsize=(13, 6))
    sns.lineplot(data=daily, x='Date', y='AQI_30d', hue='City')
    plt.title('30-Day Rolling Mean of AQI'); plt.xlabel('Date'); plt.ylabel('AQI (rolling mean)')
    plt.tight_layout(); plt.show()

## 7. Graph 5 — Monthly average AQI heatmap

Rows are cities and columns are calendar months. This combines all years in the dataset, so it reveals typical monthly patterns but not year-to-year differences.

In [ ]:
if 'AQI' in analysis.columns:
    analysis['Month'] = analysis['Date'].dt.month
    monthly = analysis.pivot_table(index='City', columns='Month', values='AQI', aggfunc='mean', observed=True)
    plt.figure(figsize=(12, 4.5)); sns.heatmap(monthly, annot=True, fmt='.0f', cmap='YlOrRd')
    plt.title('Mean AQI by City and Month (All Available Years)'); plt.xlabel('Month'); plt.ylabel('City')
    plt.tight_layout(); plt.show()

## 8. Graph 6 — Monthly AQI trend

This plot separates each year-month combination and helps show whether patterns repeat across years.

In [ ]:
if 'AQI' in analysis.columns:
    analysis['YearMonth'] = analysis['Date'].dt.to_period('M').astype(str)
    monthly_trend = analysis.groupby(['YearMonth', 'City'], as_index=False, observed=True)['AQI'].mean()
    plt.figure(figsize=(14, 6)); sns.lineplot(data=monthly_trend, x='YearMonth', y='AQI', hue='City', marker='o')
    plt.title('Monthly Mean AQI by City'); plt.xlabel('Year-Month'); plt.ylabel('Mean AQI')
    plt.xticks(rotation=70); plt.tight_layout(); plt.show()

## 9. Graph 7 — Pollutant distributions

These box plots compare the distributions of available pollutant measurements. Concentrations and units may differ by pollutant, so do not compare raw heights across pollutants as if they shared a scale.

In [ ]:
if pollutants:
    long_poll = analysis.melt(id_vars=['City', 'Date'], value_vars=pollutants, var_name='Pollutant', value_name='Concentration').dropna()
    g = sns.catplot(data=long_poll, x='Pollutant', y='Concentration', col='City', col_wrap=2, kind='box', sharey=False, height=4, aspect=1.3, showfliers=False)
    g.set_xticklabels(rotation=45); g.fig.suptitle('Pollutant Distributions by City', y=1.02); plt.show()

## 10. Graph 8 — PM2.5 versus PM10

The scatter plot shows whether days with high PM2.5 also tend to have high PM10. This is an association, not proof that one causes the other.

In [ ]:
if {'PM2.5', 'PM10'}.issubset(analysis.columns):
    sample = analysis.dropna(subset=['PM2.5', 'PM10'])
    plt.figure(); sns.scatterplot(data=sample, x='PM2.5', y='PM10', hue='City', alpha=.55)
    plt.title('PM2.5 vs PM10'); plt.tight_layout(); plt.show()
else:
    print('PM2.5 and/or PM10 unavailable.')

## 11. Graph 9 — Pollutant correlation heatmap

Correlation ranges from -1 to +1 and describes linear association between paired values. Correlation does not establish causation.

In [ ]:
corr_cols = [c for c in pollutants if analysis[c].notna().sum() > 10]
if len(corr_cols) >= 2:
    plt.figure(figsize=(10, 7)); sns.heatmap(analysis[corr_cols].corr(), annot=True, fmt='.2f', cmap='vlag', center=0, vmin=-1, vmax=1)
    plt.title('Correlation Between Pollutant Measurements'); plt.tight_layout(); plt.show()

## 12. Graph 10 — AQI category counts

The dataset's AQI_Bucket labels can be used to compare the recorded categories. Category definitions are inherited from the source dataset.

In [ ]:
if 'AQI_Bucket' in analysis.columns:
    cat = analysis.dropna(subset=['AQI_Bucket']).groupby(['City', 'AQI_Bucket'], observed=True).size().reset_index(name='Days')
    plt.figure(figsize=(12, 6)); sns.barplot(data=cat, x='City', y='Days', hue='AQI_Bucket')
    plt.title('Recorded AQI Category Counts by City'); plt.xticks(rotation=20)
    plt.tight_layout(); plt.show()

## 13. Graph 11 — Day-of-week pattern

This checks whether recorded AQI varies by weekday. Differences may reflect many factors; the plot alone cannot identify the cause.

In [ ]:
if 'AQI' in analysis.columns:
    analysis['DayOfWeek'] = analysis['Date'].dt.day_name()
    order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    weekday = analysis.groupby(['City','DayOfWeek'], observed=True)['AQI'].mean().reset_index()
    weekday['DayOfWeek'] = pd.Categorical(weekday['DayOfWeek'], categories=order, ordered=True)
    plt.figure(figsize=(13, 6)); sns.lineplot(data=weekday, x='DayOfWeek', y='AQI', hue='City', marker='o')
    plt.title('Mean AQI by Day of Week'); plt.xticks(rotation=25); plt.tight_layout(); plt.show()

## 14. Graph 12 — Highest recorded AQI days

This identifies extreme recorded values for further inspection. It does not imply that the dataset captures every pollution episode.

In [ ]:
if 'AQI' in analysis.columns:
    top_days = analysis.dropna(subset=['AQI']).nlargest(15, 'AQI')[['City','Date','AQI']].sort_values('AQI')
    display(top_days)
    plt.figure(figsize=(10, 7)); sns.barplot(data=top_days, x='AQI', y=top_days['City'] + ' — ' + top_days['Date'].dt.strftime('%Y-%m-%d'))
    plt.title('15 Highest Recorded AQI Observations'); plt.xlabel('AQI'); plt.ylabel('City — Date')
    plt.tight_layout(); plt.show()

## 15. Summary statistics and findings

Use the printed tables to write data-backed conclusions after running the notebook. Do not assume a city is worst or best before checking the results.

In [ ]:
summary_cols = [c for c in ['AQI'] + pollutants if c in analysis.columns]
display(analysis.groupby('City', observed=True)[summary_cols].agg(['count','mean','median','std','min','max']).round(2))
print('Notebook analysis complete. Interpret findings in light of missing values, station coverage, and the dataset date range.')